# EDA — rendement (t/ha)

Données **SYNTHÉTIQUES**. Checklist : docs/08_competition_ia.md §2 (EDA) et §3 (fuites).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
ML_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ML_ROOT / "src"))
from agriml.adversarial import adversarial_validation
DATA = ML_ROOT / "data/synthetic/yield"
TARGET, GROUP, TIME, ID = "yield_t_ha", "field_id", "year", "row_id"
train = pd.read_csv(DATA / "yield_train.csv")
test = pd.read_csv(DATA / "yield_test.csv")
print(train.shape, test.shape)

## 1. Forme, types, valeurs manquantes

In [ ]:
summary = pd.DataFrame({'dtype': train.dtypes.astype(str), 'na_train': train.isna().mean(),
                        'na_test': test.reindex(columns=train.columns).isna().mean(),
                        'n_unique': train.nunique()})
summary.sort_values('na_train', ascending=False).head(20)

## 2. Cible : distribution, asymétrie, valeurs extrêmes

In [ ]:
print(train[TARGET].describe())
print('asymétrie', round(train[TARGET].skew(), 3))
ax = train[TARGET].hist(bins=40); ax.set_title(TARGET); plt.close()

## 3. Groupes et temps : choix du schéma de validation croisée

In [ ]:
print('groupes train', train[GROUP].nunique(), '| test', test[GROUP].nunique(),
      '| partagés', len(set(train[GROUP]) & set(test[GROUP])))
print('périodes train', sorted(train[TIME].unique())[-3:], '| test', sorted(test[TIME].unique()))
train.groupby(TIME)[TARGET].agg(['mean', 'std', 'count']).tail(10)

## 4. Audit de fuite

Identifiants communs, doublons, variables quasi égales à la cible.

In [ ]:
print('ID communs train/test :', len(set(train[ID]) & set(test[ID])))
feats = [c for c in train.columns if c not in (TARGET, ID)]
print('doublons (hors ID/cible) :', int(train.duplicated(subset=feats).sum()))
num = train.select_dtypes('number').drop(columns=[TARGET], errors='ignore')
corr = num.corrwith(train[TARGET]).abs().sort_values(ascending=False)
suspects = corr[corr > 0.95]
print('corrélation |r| > 0.95 :', list(suspects.index))
corr.head(10)

## 5. Validation adversariale train/test

AUC ≈ 0,5 : distributions proches ; > 0,7 : dérive à traiter (pondération, retrait de variables, CV qui imite le test).

In [ ]:
# la colonne temps sépare trivialement train et test (AUC = 1) : on l'exclut de la comparaison
cols = [c for c in num.columns if c in test.columns and c not in (ID, TIME)]
adv = adversarial_validation(train, test, cols, n_splits=5)
print('AUC adversariale', round(adv.auc, 4))
pd.DataFrame(adv.top_features, columns=['variable', 'importance'])